In [1]:

#Step 1: Import Libraries
# Numerical computations
import numpy as np

# Text preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Neural network layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D
from tensorflow.keras.layers import Dense



In [2]:
#Step 2: Create Training Data

# Query-document pairs

pairs = [

"machine learning machine learning tutorial",
"machine learning deep learning concepts",
"machine learning pizza recipe",

"information retrieval search engine ranking",
"information retrieval chocolate cake recipe",

"neural networks cnn for text classification",
"neural networks football match highlights",

"transformers bert ranking model",
"transformers cooking tips"
]

# Relevance labels
# 1 = Relevant
# 0 = Not Relevant

labels = [1, 1, 0, 1, 0, 1, 0, 1, 0 ]


In [3]:
#Step 3: Convert Text to Numbers
# Create tokenizer

tokenizer = Tokenizer(num_words=1000)

# Learn vocabulary

tokenizer.fit_on_texts(pairs)

# Convert words into integer IDs

sequences = tokenizer.texts_to_sequences(pairs)

print("Word Index:")
print(tokenizer.word_index)

print("\nSequences:")
print(sequences)

#-	CNNs cannot process raw text.
#-	Every word must first be converted into a number.


Word Index:
{'learning': 1, 'machine': 2, 'recipe': 3, 'information': 4, 'retrieval': 5, 'ranking': 6, 'neural': 7, 'networks': 8, 'transformers': 9, 'tutorial': 10, 'deep': 11, 'concepts': 12, 'pizza': 13, 'search': 14, 'engine': 15, 'chocolate': 16, 'cake': 17, 'cnn': 18, 'for': 19, 'text': 20, 'classification': 21, 'football': 22, 'match': 23, 'highlights': 24, 'bert': 25, 'model': 26, 'cooking': 27, 'tips': 28}

Sequences:
[[2, 1, 2, 1, 10], [2, 1, 11, 1, 12], [2, 1, 13, 3], [4, 5, 14, 15, 6], [4, 5, 16, 17, 3], [7, 8, 18, 19, 20, 21], [7, 8, 22, 23, 24], [9, 25, 6, 26], [9, 27, 28]]


In [5]:

#Step 4: Pad Sequences
# All inputs must have the same length

max_length = 10

X = pad_sequences(
sequences,
maxlen=max_length,
padding='post'
)

# Convert labels to NumPy array

y = np.array(labels)
print(X)
#Example:
#Plain Text
#[1 2 1 2 10 11 0 0 0 0]
#Padding uses zeros to make all inputs equal length.


[[ 2  1  2  1 10  0  0  0  0  0]
 [ 2  1 11  1 12  0  0  0  0  0]
 [ 2  1 13  3  0  0  0  0  0  0]
 [ 4  5 14 15  6  0  0  0  0  0]
 [ 4  5 16 17  3  0  0  0  0  0]
 [ 7  8 18 19 20 21  0  0  0  0]
 [ 7  8 22 23 24  0  0  0  0  0]
 [ 9 25  6 26  0  0  0  0  0  0]
 [ 9 27 28  0  0  0  0  0  0  0]]


In [6]:

# Step 5: Build CNN Ranking Model
model = Sequential([

# Embedding layer
# Converts every word ID into a dense vector

Embedding(
input_dim=1000,
output_dim=50,
input_length=max_length
),

# CNN layer
# 32 filters
# Window size = 3 words

Conv1D(
filters=32,
kernel_size=3,
activation='relu'
),

# Keep strongest activation
GlobalMaxPooling1D(),

# Hidden layer

Dense(
16,
activation='relu'
),

# Output layer
# Probability that document is relevant

Dense(
1,
activation='sigmoid'
)

])

model.summary()


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
#Step 6: Compile Model
model.compile(

optimizer='adam',

loss='binary_crossentropy',

metrics=['accuracy']

)
# Relevant (1)
# Not Relevant (0)


In [8]:
#Step 7: Train Model
history = model.fit(

X,
y,

epochs=20,

verbose=1

)
##-	During training, the CNN learns important patterns indicating relevance.
##Examples of possible learned patterns:
##Plain Text
##machine learning
##search engine
##bert ranking
##cnn classification
##Non-relevant patterns:
##Plain Text
##pizza recipe
##cake recipe
##football match



Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3333 - loss: 0.6983
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.4444 - loss: 0.6923
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.8889 - loss: 0.6876
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8889 - loss: 0.6841
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 1.0000 - loss: 0.6804
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 1.0000 - loss: 0.6769
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 1.0000 - loss: 0.6735
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 1.0000 - loss: 0.6699
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 1.0000 - loss: 0.6660
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 1.0000 - loss: 0.6614
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 1.0000 - loss: 0.6562
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 1.0000 - loss: 0.6513
Epo

In [10]:
#Step 8: Test the Ranking Model
test_pairs = [

"machine learning introduction to machine learning",

"machine learning pizza delivery service",

"information retrieval search engine indexing",

"transformers bert reranking model"
]

# Convert text to numbers

test_sequences = tokenizer.texts_to_sequences(
test_pairs
)

# Pad sequences

test_data = pad_sequences(
test_sequences,
maxlen=max_length,
padding='post'
)

# Predict relevance scores

scores = model.predict(
test_data
)





1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step


In [13]:
#Step 9: Display Ranking Scores

for text, score in zip(test_pairs, scores):

  print("\nPair:")
  print(text)

  print("Relevance Score:",
  round(score[0], 4))




Pair:
machine learning introduction to machine learning
Relevance Score: 0.5307

Pair:
machine learning pizza delivery service
Relevance Score: 0.4897

Pair:
information retrieval search engine indexing
Relevance Score: 0.5254

Pair:
transformers bert reranking model
Relevance Score: 0.5188


In [16]:
#Step 10: Rank Documents for a Query
query = "machine learning"

documents = [

"machine learning tutorial",

"deep learning concepts",

"pizza recipe",

"machine learning course",

"football highlights"
]

# Combine query and document

pairs = []

for doc in documents:

  pairs.append(query + " " + doc)

# Convert to sequences

seq = tokenizer.texts_to_sequences(
pairs
)

# Pad sequences

X_test = pad_sequences(
seq,
maxlen=max_length,
padding='post'
)

# Predict relevance

scores = model.predict(X_test)

# Create ranking list

ranking = list(zip(documents, scores))

# Sort by score descending

ranking.sort(
key=lambda x: x[1],
reverse=True
)

print("\nRanking Results:")

for doc, score in ranking:

  print(doc,"->",round(score[0],4))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step

Ranking Results:
machine learning tutorial -> 0.5449
deep learning concepts -> 0.5432
machine learning course -> 0.5307
football highlights -> 0.477
pizza recipe -> 0.4494
